<a href="https://colab.research.google.com/github/Saiji/Data-Science-Work/blob/master/Virtual_Metrology_(VM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import ks_2samp

# ==========================================
# 1. SYNTHETIC DATA GENERATION (The "Fab" Simulator)
# ==========================================
def generate_fab_data(n_wafers=500, drift=False):
    np.random.seed(42)

    # Sensor data (X): Pressure, Temperature, Gas Flow, RF Power
    # In reality, these would be aggregated "Step Means" from your FDC system
    data = {
        'wafer_id': np.arange(n_wafers),
        'sensor_press_avg': np.random.normal(15.2, 0.5, n_wafers),
        'sensor_temp_max': np.random.normal(250.5, 2.1, n_wafers),
        'sensor_gas_flow': np.random.normal(120.1, 5.0, n_wafers),
        'sensor_rf_power': np.random.normal(1000.0, 10.0, n_wafers),
    }
    df = pd.DataFrame(data)

    # Physics-based Target (Y): Wafer Thickness in Angstroms
    # Thickness = f(Pressure, Temp, Gas) + Noise
    df['actual_thickness'] = (
        (df['sensor_press_avg'] * 10) +
        (df['sensor_temp_max'] * 0.8) +
        (df['sensor_gas_flow'] * 0.5) +
        np.random.normal(0, 1.5, n_wafers)
    )

    # Simulate a "Drift" event (e.g., Chamber wall polymer buildup)
    if drift:
        df['sensor_press_avg'] += 1.5  # Pressure sensor starts drifting high
        df['actual_thickness'] += 8.0  # Resulting in thicker films

    return df

# ==========================================
# 2. VIRTUAL METROLOGY MODEL (Quantile Regression)
# ==========================================
class VirtualMetrologyEngine:
    def __init__(self):
        # We use Gradient Boosting with 'quantile' loss to show confidence intervals
        self.model_mid = GradientBoostingRegressor(loss='squared_error', n_estimators=100)
        self.features = ['sensor_press_avg', 'sensor_temp_max', 'sensor_gas_flow', 'sensor_rf_power']

    def train(self, train_df):
        X = train_df[self.features]
        y = train_df['actual_thickness']
        self.model_mid.fit(X, y)
        print("✅ Model Training Complete.")

    def predict_with_confidence(self, test_df):
        preds = self.model_mid.predict(test_df[self.features])
        # In a real demo, you'd add Lower/Upper bounds here
        return preds

# ==========================================
# 3. THE "UNIQUE TWIST": DRIFT DETECTION (KS-Test)
# ==========================================
def check_for_drift(reference_data, current_data, feature):
    # Kolmogorov-Smirnov test to check if distributions have shifted
    stat, p_value = ks_2samp(reference_data[feature], current_data[feature])
    is_drifting = p_value < 0.05
    return is_drifting, p_value

# ==========================================
# 4. EXECUTION & DEMO VISUALIZATION
# ==========================================

# Step A: Setup Data
train_data = generate_fab_data(n_wafers=400)
new_production_data = generate_fab_data(n_wafers=50, drift=True) # Incoming data has drift!

# Step B: Train VM Engine
vm_engine = VirtualMetrologyEngine()
vm_engine.train(train_data)

# Step C: Real-time Prediction
predictions = vm_engine.predict_with_confidence(new_production_data)
new_production_data['predicted_thickness'] = predictions

# Step D: Drift Alert System
print("\n--- 🛡️ DRIFT DETECTION SYSTEM ---")
for feat in vm_engine.features:
    drifting, p = check_for_drift(train_data, new_production_data, feat)
    status = "🚨 ALERT: DRIFT DETECTED" if drifting else "🟢 STABLE"
    print(f"{feat:20}: {status} (p={p:.4f})")

# Step E: Visualization for Management
plt.figure(figsize=(12, 6))
plt.plot(new_production_data['wafer_id'], new_production_data['actual_thickness'], 'ko-', label='Physical Metrology (Ground Truth)')
plt.plot(new_production_data['wafer_id'], new_production_data['predicted_thickness'], 'b--', label='Virtual Metrology (Prediction)')
plt.fill_between(new_production_data['wafer_id'],
                 new_production_data['predicted_thickness'] - 2,
                 new_production_data['predicted_thickness'] + 2, color='blue', alpha=0.2, label='95% Confidence')

plt.title('Virtual Metrology Performance vs. Physical Measurement', fontsize=14)
plt.xlabel('Wafer ID')
plt.ylabel('Thickness (Å)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("\n📈 DASHBOARD INSIGHT:")
print(f"Current VM Mean Absolute Error: {mean_absolute_error(new_production_data['actual_thickness'], predictions):.2f} Å")
print(f"R-Squared Score: {r2_score(new_production_data['actual_thickness'], predictions):.4f}")